## OpenR1 Data Curation: At a Glance

Here is exactly how and why we filtered the dataset down to ~6.6k rows:

* **Strict 2:1 Ratio (FRQs vs. MCQs):** We forced a 33.3% MCQ density across all math subjects to perfectly match our test set. We used dynamic rounding to fix floating-point math errors so the quotas aligned perfectly.
* **100% Correct & Complete:** We ran a dual-check. A row was only kept if the final math was verified as correct *and* the model's internal `<think>` trace didn't cut off early. 
* **The "Double-Box" Fix:** The original dataset already had `\boxed{value}` at the end of the traces. If we just appended our own answer, the model would output two boxes. For FRQs, we just kept the original trace. For MCQs, we used a custom script to surgically swap the value inside the final box with the correct A-Z letter.
* **No Truncation (16k Token Limit):** We tokenized every prompt using the exact `Qwen3-4B-Thinking` tokenizer. We threw out any trace longer than 16,384 tokens (~1.4% of the data) so the model never learns to ramble and cut off without giving an answer.
* **Quality Over Quantity:** We ended up with 6,674 samples instead of 7,500 because the dataset ran out of valid MCQs. We chose to stop there rather than diluting the dataset with extra FRQs, which would have ruined our strict 2:1 ratio.

In [ ]:
#pip install datasets pandas

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 15.4 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 15.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 21.2 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 10.8 MB/s  0:00:03m0:00:0100:01
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  9/33 [idna]ow]
    Found exist

In [ ]:
#pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import re

def map_answer_to_letter(problem_text, answer):
    """
    Parses the problem text for MCQ options (A-Z) and matches the explicit 
    answer value to its corresponding option letter, handling LaTeX arrays.
    """
    answer_str = str(answer).strip()
    
    # 1. Check if the answer is already just a single letter (A-Z)
    if len(answer_str) == 1 and answer_str.upper() in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        return answer_str.upper()
        
    # 2. Define regex patterns for standard and LaTeX MCQ formats
    patterns = [
        # Matches (A), \textbf{(A)}, or \textbf{(A) } 
        # Captures the value up to the next option, &, \\, or end of string.
        r"(?:\\textbf\{)?\(([A-Z])\)(?:\}\s*)?(.*?)(?=(?:\\textbf\{)?\([A-Z]\)|\b[A-Z]\.|\&|\\\\|$)",
        
        # Matches standard A. format with the same boundary lookaheads
        r"\b([A-Z])\.\s*(.*?)(?=(?:\\textbf\{)?\([A-Z]\)|\b[A-Z]\.|\&|\\\\|$)"
    ]
    
    # Clean the target answer (strip $ to handle LaTeX math-mode mismatches)
    target = answer_str.strip('$').strip()
    
    # 3. Hunt for the options in the text
    for pattern in patterns:
        matches = re.findall(pattern, problem_text, re.DOTALL)
        if matches:
            for letter, option_text in matches:
                # Clean up the extracted option text (strip trailing whitespace and LaTeX artifacts)
                clean_option = option_text.strip('$&\\ \n\t')
                
                # Check for an exact match or if the target is neatly inside the option
                if target == clean_option or target in clean_option:
                    return letter.upper()
                    
    # Fallback: if we somehow can't match it, return the original answer
    return answer_str

In [6]:
from transformers import AutoTokenizer
from datasets import load_dataset

# 1. Initialize your specific Qwen tokenizer
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load the dataset in streaming mode
dataset = load_dataset("open-r1/OpenR1-Math-220k", split="train", streaming=True)

MAX_TOKENS = 16384
longest_trace = 0
over_limit_count = 0
total_checked = 0

# Adjust this if you want to scan more or fewer rows
max_to_check = 10000

print(f"Scanning dataset for token lengths using {MODEL_ID}...\n")

for row in dataset:
    generations = row.get("generations", [])
    correctness = row.get("correctness_math_verify", [])
    complete = row.get("is_reasoning_complete", [])
    
    # Check for valid generations using your established logic
    if isinstance(generations, list) and len(generations) > 0:
        valid_idx = -1
        for i in range(len(generations)):
            if i < len(correctness) and i < len(complete):
                is_correct = correctness[i] in [True, 'true', 'True']
                is_complete = complete[i] in [True, 'true', 'True']
                
                if is_correct and is_complete:
                    valid_idx = i
                    break
        
        # If we found a correct and complete trace, test its length
        if valid_idx != -1:
            reasoning = generations[valid_idx]
            answer = str(row.get("answer", ""))
            
            # Reconstruct the completion format exactly as it will appear in your data
            completion_text = f"{reasoning}\n\n\\boxed{{{answer}}}"
            
            # Tokenize and count (ignoring special system tokens for a baseline count)
            token_count = len(tokenizer.encode(completion_text, add_special_tokens=False))
            
            if token_count > longest_trace:
                longest_trace = token_count
                
            if token_count > MAX_TOKENS:
                over_limit_count += 1
                
            total_checked += 1
            
            # Progress update in the notebook output
            if total_checked % 1000 == 0:
                print(f"Checked {total_checked} valid rows... Max length so far: {longest_trace}")
                
            # Stop condition
            if total_checked >= max_to_check:
                break

print("\n--- Scan Complete ---")
print(f"Total valid traces checked: {total_checked}")
print(f"Absolute longest trace found: {longest_trace} tokens")
print(f"Traces exceeding {MAX_TOKENS} tokens: {over_limit_count}")
if total_checked > 0:
    print(f"Percentage over limit: {(over_limit_count / total_checked) * 100:.2f}%")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Scanning dataset for token lengths using Qwen/Qwen3-4B-Thinking-2507...

Checked 1000 valid rows... Max length so far: 19716
Checked 2000 valid rows... Max length so far: 19716
Checked 3000 valid rows... Max length so far: 19716
Checked 4000 valid rows... Max length so far: 19716
Checked 5000 valid rows... Max length so far: 19716
Checked 6000 valid rows... Max length so far: 19716
Checked 7000 valid rows... Max length so far: 19716
Checked 8000 valid rows... Max length so far: 22226
Checked 9000 valid rows... Max length so far: 22226
Checked 10000 valid rows... Max length so far: 22226

--- Scan Complete ---
Total valid traces checked: 10000
Absolute longest trace found: 22226 tokens
Traces exceeding 16384 tokens: 140
Percentage over limit: 1.40%


In [2]:
def swap_final_box_with_letter(reasoning_text, letter):
    """Safely finds the last \boxed{...} in a generation and replaces it with the MCQ letter."""
    idx = reasoning_text.rfind(r'\boxed{')
    if idx != -1:
        brace_count = 0
        for i in range(idx + 7, len(reasoning_text)):
            if reasoning_text[i] == '{':
                brace_count += 1
            elif reasoning_text[i] == '}':
                if brace_count == 0:
                    # Swap everything inside the final box with our letter
                    return reasoning_text[:idx] + f"\\boxed{{{letter}}}" + reasoning_text[i+1:]
                brace_count -= 1
                
    # Safe fallback just in case the original string was malformed
    return f"{reasoning_text}\n\n\\boxed{{{letter}}}"

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
import random
import json
import re
import os
from collections import defaultdict

# =========================================================
# INITIALIZE TOKENIZER FOR LENGTH FILTERING
# =========================================================

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Qwen typically uses the eos_token as the pad_token
tokenizer.pad_token = tokenizer.eos_token 
MAX_TOKENS = 16384

# =========================================================
# LOAD OPENR1 DATASET (STREAMING)
# =========================================================

dataset = load_dataset("open-r1/OpenR1-Math-220k", split="train", streaming=True)

# =========================================================
# TARGET SAMPLE SIZE & DISTRIBUTIONS
# =========================================================

TOTAL_SAMPLES = 7500

problem_distribution = {
    "Algebra": 0.50888,
    "Other": 0.18472,
    "Calculus": 0.11279,
    "Geometry": 0.10657,
    "Number Theory": 0.05329,
    "Combinatorics": 0.02487,
    "Inequalities": 0.00622,
    "Logic and Puzzle": 0.00266
}

question_distribution = {
    0: 0.666963, # FRQ
    1: 0.333037  # MCQ
}

# =========================================================
# CALCULATE NESTED TARGETS (FIXED MATH)
# =========================================================

nested_targets = {}
total_assigned = 0

for ptype, pprob in problem_distribution.items():
    total_for_type = round(pprob * TOTAL_SAMPLES)
    mcq_count = round(total_for_type * question_distribution[1])
    frq_count = total_for_type - mcq_count 
    
    nested_targets[ptype] = {
        0: frq_count,
        1: mcq_count
    }
    total_assigned += total_for_type

remainder = TOTAL_SAMPLES - total_assigned
if remainder != 0:
    nested_targets["Algebra"][0] += remainder

print("\nNested Targets:")
print(json.dumps(nested_targets, indent=2))

# =========================================================
# SYSTEM PROMPTS
# =========================================================

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

# =========================================================
# MCQ DETECTOR
# =========================================================

def is_mcq(problem_text) -> bool:
    if not isinstance(problem_text, str):
        return False
    patterns = [r"\(A\)", r"\(B\)", r"\(C\)", r"\(D\)", r"A\.", r"B\.", r"C\.", r"D\."]
    matches = sum(bool(re.search(p, problem_text)) for p in patterns)
    return matches >= 3

# =========================================================
# STORAGE
# =========================================================

sampled_rows = []
metadata_rows = []
counts = defaultdict(lambda: defaultdict(int))
skipped_length_count = 0

# =========================================================
# STREAM + STRATIFIED SAMPLE
# =========================================================

print("\nStarting dataset stream and filtering...")

for row in dataset:
    ptype = row.get("problem_type", "Other")
    if ptype not in nested_targets:
        continue

    problem_text = row.get("problem", "")
    qtype = 1 if is_mcq(problem_text) else 0

    if counts[ptype][qtype] >= nested_targets[ptype][qtype]:
        continue

    # -----------------------------------------------------
    # GET VALID REASONING TRACE
    # -----------------------------------------------------
    generations = row.get("generations", [])
    correctness_math_verify = row.get("correctness_math_verify", [])
    is_reasoning_complete = row.get("is_reasoning_complete", [])

    if isinstance(generations, list) and len(generations) > 0:
        valid_idx = -1
        for i in range(len(generations)):
            if i < len(correctness_math_verify) and i < len(is_reasoning_complete):
                is_correct = correctness_math_verify[i] in [True, 'true', 'True']
                is_complete = is_reasoning_complete[i] in [True, 'true', 'True']
                
                if is_correct and is_complete:
                    valid_idx = i
                    break 
        
        if valid_idx != -1:
            reasoning = generations[valid_idx]
        else:
            continue
    else:
        continue

    # -----------------------------------------------------
    # GET FINAL ANSWER & FORMAT
    # -----------------------------------------------------
    if qtype == 1:
        # It's an MCQ: Map the answer to a letter and swap out the original boxed value
        raw_answer = row.get("answer", "")
        answer_letter = map_answer_to_letter(problem_text, raw_answer)
        assistant_output = swap_final_box_with_letter(reasoning, answer_letter)
        system_prompt = SYSTEM_PROMPT_MCQ
    else:
        # It's an FRQ: The original reasoning is already correct and contains the \boxed{}
        assistant_output = reasoning
        system_prompt = SYSTEM_PROMPT_MATH

    # -----------------------------------------------------
    # TOKEN LENGTH FILTER
    # -----------------------------------------------------
    # Concatenate the full text to get an accurate token count for the whole sequence
    full_text = f"{system_prompt}\n{problem_text}\n{assistant_output}"
    token_count = len(tokenizer.encode(full_text, add_special_tokens=False))
    
    if token_count > MAX_TOKENS:
        skipped_length_count += 1
        continue # Skip this row, do not count it toward the quota

    # -----------------------------------------------------
    # FORMAT & STORE
    # -----------------------------------------------------
    formatted_row = {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": problem_text}
        ],
        "completion": [
            {"role": "assistant", "content": assistant_output}
        ]
    }

    sampled_rows.append(formatted_row)
    metadata_rows.append({"problem_type": ptype, "question_type": qtype})
    counts[ptype][qtype] += 1

    if len(sampled_rows) >= TOTAL_SAMPLES:
        break

# =========================================================
# SHUFFLE & SAVE
# =========================================================

random.shuffle(sampled_rows)

output_path = "data/openr1_math_7_5k_stratified.jsonl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w") as f:
    for row in sampled_rows:
        f.write(json.dumps(row) + "\n")

print(f"\nSaved dataset to:\n{output_path}")

# VERIFY DISTRIBUTIONS
metadata_df = pd.DataFrame(metadata_rows)
print("\nProblem Type Distribution:")
print(metadata_df["problem_type"].value_counts(normalize=True))
print("\nQuestion Type Distribution:")
print(metadata_df["question_type"].value_counts(normalize=True))
print("\nTotal Samples Saved:")
print(len(metadata_df))
print(f"\nTotal Traces Skipped Due to Length (> {MAX_TOKENS} tokens): {skipped_length_count}")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]


Nested Targets:
{
  "Algebra": {
    "0": 2545,
    "1": 1271
  },
  "Other": {
    "0": 924,
    "1": 461
  },
  "Calculus": {
    "0": 564,
    "1": 282
  },
  "Geometry": {
    "0": 533,
    "1": 266
  },
  "Number Theory": {
    "0": 267,
    "1": 133
  },
  "Combinatorics": {
    "0": 125,
    "1": 62
  },
  "Inequalities": {
    "0": 31,
    "1": 16
  },
  "Logic and Puzzle": {
    "0": 13,
    "1": 7
  }
}

Starting dataset stream and filtering...

Saved dataset to:
data/openr1_math_7_5k_stratified.jsonl

Problem Type Distribution:
problem_type
Algebra          0.603511
Geometry         0.126364
Calculus         0.094417
Other            0.075439
Number Theory    0.063261
Combinatorics    0.029575
Inequalities     0.007433
Name: proportion, dtype: float64

Question Type Distribution:
question_type
0    0.708208
1    0.291792
Name: proportion, dtype: float64

Total Samples Saved:
6323

Total Traces Skipped Due to Length (> 16384 tokens): 62


In [4]:
new_new = pd.read_json('data/openr1_math_7_5k_stratified.jsonl', lines=True)

In [5]:
pd.set_option('display.max_colwidth', None)
display(new_new.iloc[[8]])

prompt  \
8  [{'role': 'system', 'content': 'You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.'}, {'role': 'user', 'content': 'Princeton has an endowment of $5$ million dollars and wants to invest it into improving campus life. The university has three options: it can either invest in improving the dorms, campus parties or dining hall food quality. If they invest $a$ million dollars in the dorms, the students will spend an additional $5a$ hours per week studying. If the university invests $b$ million dollars in better food, the students will spend an additional $3b$ hours per week studying. Finally, if the $c$ million dollars are invested in parties, students will be more relaxed and spend $11c - c^2$ more hours per week studying. The university wants to invest its $5$ million dollars so that the students get as many additional hours of studying as possible. What is the maximal amount that students get to study?'}]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       